### Re-annotate the three pre-built `.h5ad` files for the LGR5 cohort

- **Developed by:** Anna Maguza
- **Affiliation:** Faculty of Medicine, Würzburg University
- **Date of creation:** 7 May 2026
- **Last modified date:** 7 May 2026

Three AnnData files already exist on disk and are not re-created from raw counts here — they are loaded, **augmented in place** with the cohort-wide `obs` columns (`technology`, `lgr5_status`, `lgr5_label_raw`, `assay_modality`, `GSE`, `organism`), then saved under the canonical filename pattern `data/LGR5_analysis/gut_{hs|mm}_<study>_AM_<ts>_raw.h5ad`. Existing obs / var / layers / uns are preserved.

| source file (under `LGR5_analysis_data/`) | study | shape | source obs |
|---|---|---|---|
| `Grün_2016_raw.h5ad` | Grün 2015 (GSE62270 primary intestine 96-well plate) | 96 × 57186 | empty |
| `Haber_2017_Smartseq_LGR5_FACS_data_TPM_normalized.h5ad` | Haber 2017 Smart-Seq2 | 1522 × 20107 | `barcode`, `Gene_marker`, `Donor_ID`, `GFP_intensity`, `Cell_Type` |
| `Processed/Ishikawa_2022/Ishikawa_2022_raw_filtered.h5ad` | Ishikawa 2022 healthy ascending colon | 7103 × 33538 | `cell_id`, `celltype`, `obsm.X_umap` |

LGR5 mapping decisions (from `LGR5_data_folder_inventory.md` + GSE_datasets entries 11/13/Ishikawa):
- **Grün 2015**: every cell is FACS-sorted Lgr5-EGFP+ → `lgr5_status='LGR5+'` uniformly.
- **Haber 2017**: `GFPHigh` → `LGR5+` (Lgr5-EGFP-high); `GFPLow` → `LGR5-` (Lgr5-EGFP-low daughters — the existing repo notebooks treat these as the LGR5- comparator, see `data/Haber_2017_Smartseq/gut_mm_DEG_Haber2017_LGR5+_vs_Haber2017_LGR5-_*.csv`); the three `GFP-.*` gates → `LGR5-` with the sub-gate (`CD24+`, `CD24+.cKit+`, `CD24-.EpCAM+`) preserved in `lgr5_label_raw`.
- **Ishikawa 2022**: `celltype == 'Stem'` → `LGR5+` (Stem cluster is the Lgr5+ ISC cluster per the original Ishikawa annotation); all other celltypes → `unknown` (they are not Lgr5-sorted, just other epithelial cell types).

### Import packages

In [1]:
import os
from datetime import datetime

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc

/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [2]:
OUT_DIR = 'data/LGR5_analysis'
os.makedirs(OUT_DIR, exist_ok=True)
TS = datetime.now().strftime('%d%m%Y_%H%M%S')

In [3]:
def merge_history(uns_history, new_ts, new_step):
    """Coerce existing uns['processing_history'] (dict | list | missing) → dict[ts→step] then add the new entry."""
    out = {}
    if uns_history is None:
        pass
    elif isinstance(uns_history, dict):
        if 'timestamp' in uns_history and 'step' in uns_history:
            out[str(uns_history['timestamp'])] = str(uns_history['step'])
        else:
            out = {str(k): str(v) for k, v in uns_history.items()}
    elif isinstance(uns_history, (list, tuple)):
        for i, entry in enumerate(uns_history):
            if isinstance(entry, dict) and 'timestamp' in entry and 'step' in entry:
                out[str(entry['timestamp'])] = str(entry['step'])
            else:
                out[f'step_{i:02d}'] = str(entry)
    else:
        out['legacy'] = str(uns_history)
    out[new_ts] = new_step
    return out

### 1. Grün 2016 → re-annotate as `gut_mm_Grün2016_AM_<ts>_raw.h5ad`

In [4]:
src = '/Users/am336941/PhD/data/LGR5_analysis_data/Grün_2016_raw.h5ad'
a = sc.read_h5ad(src)
a.obs['cell_id']        = a.obs.index.astype(str)
a.obs['sample']         = 'Lgr5_primary_GSE62270'
a.obs['lgr5_status']    = 'LGR5+'
a.obs['lgr5_label_raw'] = 'Lgr5-EGFP+'
a.obs['condition']      = 'Lgr5-EGFP+'
a.obs['cell_type']      = 'primary Lgr5+ ISC'
a.obs['tissue']         = 'small intestine'
a.obs['GSE']            = 'GSE62270'
a.obs['organism']       = 'mus musculus'
a.obs['technology']     = 'CEL-Seq'
a.obs['assay_modality'] = 'single-cell'
for col in ['sample', 'lgr5_status', 'lgr5_label_raw', 'condition', 'cell_type', 'tissue', 'GSE', 'organism', 'technology', 'assay_modality']:
    a.obs[col] = a.obs[col].astype('category')

a.uns['processing_history'] = merge_history(
    a.uns.get('processing_history'), TS,
    f're-annotated existing Grün_2016_raw.h5ad with cohort-wide obs columns (technology, lgr5_status, lgr5_label_raw, assay_modality, GSE, organism, tissue); no X / var changes; source={src}',
)
a.uns['GSE'] = 'GSE62270'
a.uns['publication'] = 'Grün D et al., Nature 525:251-255 (2015) — RaceID single-cell mRNA-seq'
a.uns['genome_reference'] = 'mm9 (per Grün 2015)'

out = f'{OUT_DIR}/gut_mm_Grün2016_AM_{TS}_raw.h5ad'
a.write_h5ad(out)
print(out, a.shape, '|', dict(a.obs[['lgr5_status', 'sample']].value_counts()))

data/LGR5_analysis/gut_mm_Grün2016_AM_07052026_230357_raw.h5ad (96, 57186) | {('LGR5+', 'Lgr5_primary_GSE62270'): np.int64(96)}


### 2. Haber 2017 → re-annotate as `gut_mm_Haber2017_AM_<ts>_TPM.h5ad`

In [5]:
src = '/Users/am336941/PhD/data/LGR5_analysis_data/gut_mm_Haber2017_SmartSeq2_QCfiltered_AM_15112024_113549_raw.h5ad'
a = sc.read_h5ad(src)

GFP_TO_LGR5 = {
    'GFPHigh':            ('LGR5+', 'Lgr5-EGFP-high'),
    'GFPLow':             ('LGR5-', 'Lgr5-EGFP-low'),
    'GFP-.CD24+':         ('LGR5-', 'Lgr5-EGFP-neg.CD24+'),
    'GFP-.CD24+.cKit+':   ('LGR5-', 'Lgr5-EGFP-neg.CD24+.cKit+'),
    'GFP-.CD24-.EpCAM+':  ('LGR5-', 'Lgr5-EGFP-neg.CD24-.EpCAM+'),
}
missing = set(a.obs['GFP_intensity'].unique()) - set(GFP_TO_LGR5)
if missing:
    raise ValueError(f'unmapped GFP_intensity values: {missing}')

a.obs['lgr5_status']    = a.obs['GFP_intensity'].map(lambda v: GFP_TO_LGR5[v][0])
a.obs['lgr5_label_raw'] = a.obs['GFP_intensity'].map(lambda v: GFP_TO_LGR5[v][1])
a.obs['sample']         = a.obs['Donor_ID'].astype(str) + '_' + a.obs['GFP_intensity'].astype(str)
a.obs['condition']      = a.obs['GFP_intensity'].astype(str)
a.obs['cell_type']      = a.obs['Cell_Type'].astype(str)
a.obs['tissue']         = 'small intestine'
a.obs['GSE']            = 'GSE92332'
a.obs['organism']       = 'mus musculus'
a.obs['technology']     = 'Smart-Seq2'
a.obs['assay_modality'] = 'single-cell'
for col in ['sample', 'lgr5_status', 'lgr5_label_raw', 'condition', 'cell_type', 'tissue', 'GSE', 'organism', 'technology', 'assay_modality']:
    a.obs[col] = a.obs[col].astype('category')

a.uns['processing_history'] = merge_history(
    a.uns.get('processing_history'), TS,
    f're-annotated Haber_2017_Smartseq_LGR5_FACS_data_TPM_normalized.h5ad: lgr5_status mapped from GFP_intensity (GFPHigh→LGR5+, GFPLow→LGR5-, all GFP-.*→LGR5-); existing barcode/Gene_marker/Donor_ID/GFP_intensity/Cell_Type retained; source={src}',
)
a.uns['GSE'] = 'GSE92332'
a.uns['publication'] = 'Haber AL et al., Nature 551:333-339 (2017) — A single-cell survey of the small intestinal epithelium (Smart-Seq2)'
a.uns['genome_reference'] = 'mm10 (per Haber 2017; values are raw - were remapped with StarSolo)'

out = f'{OUT_DIR}/gut_mm_Haber2017_AM_{TS}_raw.h5ad'
a.write_h5ad(out)
print(out, a.shape)
print('lgr5_status × Cell_Type:')
print(a.obs.groupby(['lgr5_status', 'Cell_Type'], observed=True).size())

/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


data/LGR5_analysis/gut_mm_Haber2017_AM_15052026_122932_raw.h5ad (1453, 51241)
lgr5_status × Cell_Type:
lgr5_status  Cell_Type                   
LGR5+        Endocrine                         4
             Enterocyte-Progenitor-Early       1
             Goblet                            4
             LGR5+ stem cell                 351
             TA                               24
LGR5-        Endocrine                        35
             Enterocyte                      149
             Enterocyte-Progenitor-Early     111
             Enterocyte-Progenitor-Late       68
             Goblet                          114
             LGR5- CD24- EpCAM+ stem cell     51
             LGR5- stem cell                  66
             Paneth                           69
             Stem                            141
             TA                              166
             Tuft-1                           50
             Tuft-2                           49
dtype: int64


### 3. Ishikawa 2022 → re-annotate as `gut_hs_Ishikawa2022_AM_<ts>_raw.h5ad`

Source already follows the canonical filename pattern (`gut_hs_Ishikawa2022_AM_15112024_120658_raw.h5ad` exists in the same folder); the new file just adds the cohort obs columns. UMAP embedding (`obsm.X_umap`) and the existing `celltype` annotation are preserved.

In [6]:
src = '/Users/am336941/PhD/data/LGR5_analysis_data/Processed/Ishikawa_2022/Ishikawa_2022_raw_filtered.h5ad'
a = sc.read_h5ad(src)

is_stem = a.obs['celltype'].astype(str) == 'Stem'
a.obs['lgr5_status']    = np.where(is_stem, 'LGR5+', 'unknown')
a.obs['lgr5_label_raw'] = a.obs['celltype'].astype(str)
a.obs['sample']         = 'Ishikawa2022'
a.obs['condition']      = 'healthy ascending colon'
a.obs['cell_type']      = a.obs['celltype'].astype(str)
a.obs['tissue']         = 'ascending colon'
a.obs['GSE']            = 'Ishikawa2022 (data shared by authors; not via GEO)'
a.obs['organism']       = 'homo sapiens'
a.obs['technology']     = '10x Chromium'
a.obs['assay_modality'] = 'single-cell'
for col in ['sample', 'lgr5_status', 'lgr5_label_raw', 'condition', 'cell_type', 'tissue', 'GSE', 'organism', 'technology', 'assay_modality']:
    a.obs[col] = a.obs[col].astype('category')

a.uns['processing_history'] = merge_history(
    a.uns.get('processing_history'), TS,
    f're-annotated Ishikawa_2022_raw_filtered.h5ad: lgr5_status=LGR5+ for celltype==Stem cluster, unknown otherwise; existing celltype + UMAP preserved; source={src}',
)
a.uns['GSE'] = 'Ishikawa2022'
a.uns['publication'] = 'Ishikawa et al., 2022 — healthy ascending colon scRNA-seq; data shared directly by the authors'
a.uns['genome_reference'] = 'GRCh38 (per Ishikawa 2022)'

out = f'{OUT_DIR}/gut_hs_Ishikawa2022_AM_{TS}_raw.h5ad'
a.write_h5ad(out)
print(out, a.shape)
print('lgr5_status counts:', dict(a.obs.lgr5_status.value_counts()))

data/LGR5_analysis/gut_hs_Ishikawa2022_AM_07052026_230357_raw.h5ad (7103, 33538)
lgr5_status counts: {'unknown': np.int64(6995), 'LGR5+': np.int64(108)}


/Users/am336941/.local/share/uv/python/cpython-3.11.12-macos-aarch64-none/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Verify all three re-annotated files have the cohort obs columns

In [7]:
import glob
for f in sorted(glob.glob(f'{OUT_DIR}/gut_*_AM_{TS}_*.h5ad')):
    a = sc.read_h5ad(f)
    cohort_cols = ['technology', 'lgr5_status', 'lgr5_label_raw', 'assay_modality', 'GSE', 'organism']
    missing = [c for c in cohort_cols if c not in a.obs.columns]
    print(f'{f.split("/")[-1]:55s} {a.shape}  lgr5_status={dict(a.obs.lgr5_status.value_counts())}  missing={missing or "OK"}')

gut_hs_Ishikawa2022_AM_07052026_230357_raw.h5ad         (7103, 33538)  lgr5_status={'unknown': np.int64(6995), 'LGR5+': np.int64(108)}  missing=OK
gut_mm_Grün2016_AM_07052026_230357_raw.h5ad             (96, 57186)  lgr5_status={'LGR5+': np.int64(96)}  missing=OK
gut_mm_Haber2017_AM_07052026_230357_TPM.h5ad            (1522, 20107)  lgr5_status={'LGR5-': np.int64(1111), 'LGR5+': np.int64(411)}  missing=OK
